# Fine-tuning LeRobot Policy (SO-100)

Поддерживаемые модели: **ACT**, **Diffusion**, **pi0**, **pi0.5**, **SmolVLA**.
Датасет собирается через вкладку **Dataset** в Desktop App.

In [ ]:
import torch
import ray
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  |  PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Загружаем собранный датасет
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

DATASET_PATH = Path("../datasets/lerobot")
if DATASET_PATH.exists():
    dataset = LeRobotDataset(repo_id=DATASET_PATH, episodes=[0])
    print(f"Loaded {len(dataset)} frames from {DATASET_PATH}")
else:
    print("Dataset not found. Use Dataset tab to record episodes first.")
    dataset = None

## 1. Train ACT (Action Chunking with Transformers)

In [ ]:
from lerobot.policies import ACTConfig
from lerobot.policies.factory import make_policy

config = ACTConfig(
    dim_model=256,
    n_heads=8,
    dim_feedforward=3200,
    n_encoder_layers=4,
    n_decoder_layers=7,
    chunk_size=100,
)
policy = make_policy(config).to(DEVICE)
print(f"ACT Policy: {sum(p.numel() for p in policy.parameters())/1e6:.1f}M params")

In [ ]:
if dataset is not None:
    optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-4, weight_decay=1e-5)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True)

    for epoch in range(50):
        total_loss = 0.0
        for batch in dataloader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = policy.forward(batch)["loss"]
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
        print(f"Epoch {epoch+1:2d} | Loss: {total_loss/len(dataloader):.4f}")
    print("ACT training complete")

## 2. Fine-tune pi0 (Vision-Language-Action)

pi0 — foundation model от Physical Intelligence. Работает на SO-100.
Требует `pip install -e ".[pi]"`.

In [ ]:
from lerobot.common.policies.pi0 import PI0Policy

policy = PI0Policy.from_pretrained("lerobot/pi0_base")
policy = policy.to(DEVICE).eval()
print(f"pi0: {sum(p.numel() for p in policy.parameters())/1e6:.1f}M params")
print("Ready for inference. For training, use lerobot-train CLI.")

## 3. Fine-tune pi0.5

pi0.5 — improved VLA с open-world generalization.

In [ ]:
from lerobot.common.policies.pi05 import PI05Policy

policy = PI05Policy.from_pretrained("lerobot/pi05_base")
policy = policy.to(DEVICE).eval()
print(f"pi0.5: {sum(p.numel() for p in policy.parameters())/1e6:.1f}M params")

## 4. Training via CLI (рекомендуемый способ)

```bash
# ACT
lerobot-train \
    --dataset.repo_id=lerobot/so100_pick_cup \
    --policy.type=act \
    --output_dir=./outputs/act_so100 \
    --steps=5000 \
    --device=cuda

# pi0
lerobot-train \
    --dataset.repo_id=lerobot/so100_pick_cup \
    --policy.type=pi0 \
    --policy.pretrained_path=lerobot/pi0_base \
    --output_dir=./outputs/pi0_so100 \
    --steps=3000 \
    --policy.compile_model=true \
    --policy.gradient_checkpointing=true \
    --policy.dtype=bfloat16 \
    --batch_size=8

# pi0.5
lerobot-train \
    --dataset.repo_id=lerobot/so100_pick_cup \
    --policy.type=pi05 \
    --policy.pretrained_path=lerobot/pi05_base \
    --output_dir=./outputs/pi05_so100 \
    --steps=3000 \
    --policy.compile_model=true \
    --policy.gradient_checkpointing=true \
    --policy.dtype=bfloat16 \
    --batch_size=8
```

## 5. Ray Tune — Hyperparameter Optimization

Опционально: параллельный поиск гиперпараметров через Ray.

In [ ]:
import ray
from ray import tune

if not ray.is_initialized():
    ray.init(include_dashboard=False)

def train_fn(config):
    from lerobot.policies import ACTConfig
    from lerobot.policies.factory import make_policy
    
    lr = config["lr"]
    chunk_size = config["chunk_size"]
    
    cfg = ACTConfig(dim_model=256, n_heads=8, dim_feedforward=3200,
                    n_encoder_layers=4, n_decoder_layers=7,
                    chunk_size=chunk_size)
    policy = make_policy(cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(policy.parameters(), lr=lr)
    
    for step in range(10):
        loss = torch.tensor(1.0, requires_grad=True)  # TODO: real batch
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        tune.report({"loss": loss.item()})

# Раскомментируй для запуска:
# tuner = tune.Tuner(train_fn, param_space={
#     "lr": tune.loguniform(1e-5, 1e-3),
#     "chunk_size": tune.choice([50, 100, 200]),
# })
# results = tuner.fit()
# print("Best config:", results.get_best_result(metric="loss", mode="min").config)
print("Ray Tune ready. Uncomment to run.")

## 6. Save & Export

In [ ]:
# Save pytorch checkpoint
ckpt_dir = Path("checkpoints/my_policy")
ckpt_dir.mkdir(parents=True, exist_ok=True)
policy.save_pretrained(str(ckpt_dir))
print(f"Checkpoint saved to {ckpt_dir}")

# Загрузить потом можно:
# from lerobot.policies.factory import make_policy
# policy = make_policy(pretrained=str(ckpt_dir))
# policy = policy.to(DEVICE).eval()